In [1]:
import os, sys
os.chdir('/home/liangzida/workspace/iTransformer') # 更改工作目录到项目根目录
sys.path.append('/home/liangzida/workspace/iTransformer') # 添加模块路径到 sys.path
import torch
from data_provider.data_loader import Dataset_ETT_hour, Dataset_ETT_minute, Dataset_Custom, Dataset_Solar, Dataset_PEMS, \
    Dataset_Pred
from torch.utils.data import DataLoader

data_dict = {
    'ETTh1': Dataset_ETT_hour,
    'ETTh2': Dataset_ETT_hour,
    'ETTm1': Dataset_ETT_minute,
    'ETTm2': Dataset_ETT_minute,
    'Solar': Dataset_Solar,
    'PEMS': Dataset_PEMS,
    'custom': Dataset_Custom,
}


def data_provider(args, flag):
    Data = data_dict[args.data]
    timeenc = 0 if args.embed != 'timeF' else 1

    if flag == 'test':
        shuffle_flag = False
        drop_last = True
        batch_size = 1  # bsz=1 for evaluation
        freq = args.freq
    elif flag == 'pred':
        shuffle_flag = False
        drop_last = False
        batch_size = 1
        freq = args.freq
        Data = Dataset_Pred
    else:
        shuffle_flag = True
        drop_last = True
        batch_size = args.batch_size  # bsz for train and valid
        freq = args.freq

    class sub_Data(Data):
        def __init__(self, **kwargs):
            super(sub_Data, self).__init__(**kwargs)

        def __getitem__(self, index):
            channels = self.data_x.shape[1]
            s_begin = index // channels
            seq_x, seq_y, seq_x_mark, seq_y_mark = super(sub_Data, self).__getitem__(s_begin)
            channel_idx = index % channels
            seq_x = seq_x[:, channel_idx]
            seq_y = seq_y[:, channel_idx]
            return torch.tensor(seq_x), torch.tensor(seq_y), torch.tensor(seq_x_mark).expand(-1, 4), torch.tensor(seq_y_mark).expand(-1, 4)

        def __len__(self):
            return super(sub_Data, self).__len__() * self.data_x.shape[1]
    Data = sub_Data
    data_set = Data(
        root_path=args.root_path,
        data_path=args.data_path,
        flag=flag,
        size=[args.seq_len, args.label_len, args.pred_len],
        features=args.features,
        target=args.target,
        timeenc=timeenc,
        freq=freq,
    )
    print(flag, len(data_set))
    data_loader = DataLoader(
        data_set,
        batch_size=batch_size,
        shuffle=shuffle_flag,
        num_workers=args.num_workers,
        drop_last=drop_last)
    return data_set, data_loader


/home/liangzida/anaconda3/envs/ts/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# from data_provider.data_factory import data_provider

class Args():
    def __init__(self):
        self.embed = 'timeF'
        self.target = 'OT'
        self.label_len = 1
        self.freq = 'h'

        self.data = 'custom'
        self.root_path = './dataset/electricity/'
        self.data_path = 'electricity.csv'
        self.features = 'M'
        self.enc_in = 321
        self.dec_in = 321

        self.seq_len = 24
        self.pred_len = 24
        self.batch_size = 16
        self.num_workers = 0

dataset_configs = [
    {'root_path':'./dataset/traffic/', 'data_path':'traffic.csv', 'features':'M', 'enc_in':862, 'dec_in':862},
    {'root_path':'./dataset/weather/', 'data_path':'weather.csv', 'features':'M', 'enc_in':21, 'dec_in':21},
    {'root_path':'./dataset/ETT-small/', 'data_path':'ETTh1.csv', 'data':'ETTh1', 'features':'M', 'enc_in':7, 'dec_in':7},
    {'root_path':'./dataset/ETT-small/', 'data_path':'ETTh2.csv', 'data':'ETTh2', 'features':'M', 'enc_in':7, 'dec_in':7},
    {'root_path':'./dataset/ETT-small/', 'data_path':'ETTm1.csv', 'data':'ETTm1', 'features':'M', 'enc_in':7, 'dec_in':7},
    {'root_path':'./dataset/ETT-small/', 'data_path':'ETTm2.csv', 'data':'ETTm2', 'features':'M', 'enc_in':7, 'dec_in':7},
    {'root_path':'./dataset/PEMS/', 'data_path':'PEMS03.npz', 'data':'PEMS', 'features':'M', 'enc_in':358, 'dec_in':358},
    {'root_path':'./dataset/PEMS/', 'data_path':'PEMS04.npz', 'data':'PEMS', 'features':'M', 'enc_in':307, 'dec_in':307},
    {'root_path':'./dataset/PEMS/', 'data_path':'PEMS07.npz', 'data':'PEMS', 'features':'M', 'enc_in':883, 'dec_in':883},
    {'root_path':'./dataset/PEMS/', 'data_path':'PEMS08.npz', 'data':'PEMS', 'features':'M', 'enc_in':170, 'dec_in':170},
    {'root_path':'./dataset/Solar/', 'data_path':'solar_AL.txt', 'data':'Solar', 'features':'M', 'enc_in':137, 'dec_in':137},
]

datasets = []
for config in dataset_configs:
    args = Args()
    args.__dict__.update(config)
    data_set, data_loader = data_provider(args, 'train')
    datasets.append(data_set)

train 10544846
train 773640
train 60151
train 60151
train 241591
train 241591
train 5612366
train 3115436
train 14911221
train 1813220
train 5034065


In [3]:
type(datasets[0][0][0])

torch.Tensor

In [4]:
from torch.utils.data import ConcatDataset

# 将datasets合并成一个大的dataset
combined_dataset = ConcatDataset(datasets)
combined_loader = torch.utils.data.DataLoader(combined_dataset, batch_size=16, shuffle=True, num_workers=10, drop_last=True)


In [5]:
for i, data in enumerate(combined_loader):
    print(i, data)
    if i == 10:
        break

RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/liangzida/anaconda3/envs/ts/lib/python3.9/site-packages/torch/utils/data/_utils/worker.py", line 302, in _worker_loop
    data = fetcher.fetch(index)
  File "/home/liangzida/anaconda3/envs/ts/lib/python3.9/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    return self.collate_fn(data)
  File "/home/liangzida/anaconda3/envs/ts/lib/python3.9/site-packages/torch/utils/data/_utils/collate.py", line 175, in default_collate
    return [default_collate(samples) for samples in transposed]  # Backwards compatibility.
  File "/home/liangzida/anaconda3/envs/ts/lib/python3.9/site-packages/torch/utils/data/_utils/collate.py", line 175, in <listcomp>
    return [default_collate(samples) for samples in transposed]  # Backwards compatibility.
  File "/home/liangzida/anaconda3/envs/ts/lib/python3.9/site-packages/torch/utils/data/_utils/collate.py", line 140, in